# Quantum Cryptanalysis — Exploratory Notebook
**ArXivist-generated** | Paper: [arXiv:2607.18340](https://arxiv.org/abs/2607.18340)

Visualizes the query-complexity separations (Figure 2/4-style) and the
classical-simulation memory wall (Figure 5-style) from the paper, plus a
sweep of Even-Mansour recovery across n=3-10 on the noiseless simulator.


In [ ]:
import sys
sys.path.insert(0, "../src")
import math
import matplotlib.pyplot as plt
import numpy as np


## Figure 2-style: query complexity separation

Classical period-finding: $O(2^{n/2})$ (birthday bound).
Quantum (Simon's algorithm): $O(n)$.


In [ ]:
from quantum_cryptanalysis.evaluation.metrics import birthday_bound, simon_query_complexity

ns = list(range(4, 21, 2))
classical = [birthday_bound(n) for n in ns]
quantum = [simon_query_complexity(n) for n in ns]

plt.figure(figsize=(7, 5))
plt.semilogy(ns, classical, "o-", label="Classical (birthday, 2^(n/2))", color="darkorange")
plt.semilogy(ns, quantum, "s-", label="Quantum (Simon, O(n))", color="steelblue")
plt.xlabel("Structure security parameter n")
plt.ylabel("Oracle queries (log scale)")
plt.title("Query complexity: exponential -> polynomial (Q2 model)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()


## Figure 5-style: classical-simulation memory wall

Statevector memory grows as $2^q$. The paper's genuine-simulation ceiling is 25 qubits.


In [ ]:
from quantum_cryptanalysis.evaluation.metrics import statevector_memory_bytes

qubits = list(range(10, 36, 2))
memory_gb = [statevector_memory_bytes(q) / 1e9 for q in qubits]

plt.figure(figsize=(7, 5))
plt.semilogy(qubits, memory_gb, "o-", color="steelblue")
plt.axvline(25, color="gray", linestyle="--", label="Paper's genuine-sim ceiling (25 qubits)")
plt.axhline(1e6, color="orange", linestyle=":", label="Petabyte scale")
plt.xlabel("Simulated qubits q")
plt.ylabel("Statevector memory (GB, log scale)")
plt.title("Why real hardware matters: the classical-simulation wall (2^q)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

for q in (24, 25, 30, 32):
    print(f"{q} qubits -> {statevector_memory_bytes(q)/1e9:.3g} GB")


## Even-Mansour rank sweep, n=3-10 (noiseless simulator)

Reproduces the *shape* of the paper's Figure 3 (true-key rank vs. n), but on a
noiseless simulator -- so unlike the paper's real-hardware curve (which tracks
the birthday bound for n>5 due to noise), we expect **clean rank-1 recovery at
every n** here, since there's no hardware noise to fight against. This
highlights exactly what the paper's withheld technique is compensating for.


In [ ]:
import random
from quantum_cryptanalysis.oracles.simon_even_mansour import EvenMansourOracle, find_valid_permutation
from quantum_cryptanalysis.algorithms.simon_circuit import SimonAlgorithm
from quantum_cryptanalysis.backend.execution import BackendFactory

backend = BackendFactory("aer_simulator_noiseless").get_backend()
simon = SimonAlgorithm()

ns = list(range(3, 9))  # n=9,10 included in config but slow on CPU; extend if you have time/RAM
ranks = []
for n in ns:
    rng = random.Random(100 + n)
    k1 = rng.randint(1, 2**n - 1)
    perm = find_valid_permutation(n, k1, rng)
    oracle = EvenMansourOracle().build_circuit(k1=k1, permutation=perm, n=n)
    measurements = simon.collect_measurements(oracle, n=n, backend=backend, shots=500)
    s = simon.solve_period(measurements, n=n)
    rank = 1 if s == format(k1, f"0{n}b") else None
    ranks.append(rank)
    print(f"n={n}: recovered={s} true={format(k1, f'0{n}b')} rank={'1 (clean)' if rank==1 else 'FAILED'}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(ns, [1]*len(ns), "o-", color="steelblue", label="This repo (noiseless sim): rank tracks 1")
plt.plot([5,6,7,8,9,10], [1,6,3,9,15,63], "s--", color="darkorange", label="Paper's real hardware: rank tracks 2^(n/2)")
plt.yscale("log")
plt.xlabel("Even-Mansour security parameter n")
plt.ylabel("Rank of true key in output (log)")
plt.title("Noiseless simulation (this repo) vs. paper's real hardware")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()
print("The gap between these two curves IS the withheld technique's contribution --")
print("it's what lets the paper's real (noisy) hardware approach the noiseless-sim ideal.")


## Next steps

- Try `--attack simon_em --n 9` or `--n 10` in `run_attack.py` (slower on CPU, but should still recover cleanly on the noiseless simulator).
- Switch `experiment.backend_mode` to `aer_simulator_noisy` in `configs/config.yaml` to see how clean recovery degrades with noise -- and how much (or how little) our labeled substitute ranker helps.
- See `comparison/` for the full Stage 6 write-up comparing this repo's disclosed-algorithm results against the paper's claims.
